# Experiment 4: GPT-2 Small Layer-8 Residual-Stream Sweep

Proposal target: train VG-SAE on GPT-2 small residual-stream activations, sweep dictionary width and lambda, and plot `lambda*(N)=argmax_lambda V_eff(lambda,N)`.

Interpretation is deliberately narrow: `lambda*` is a recoverability-transition diagnostic, not the true number of language-model features.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / ".matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
DEVICE = torch.device("cpu")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebooks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from src.gpt2_activations import ActivationCacheConfig, cache_gpt2_residual_activations, load_activation_cache
from src.sae_evaluate import susceptibility, vg_sae_observables
from src.sae_model import VGSAEConfig, VariationalGarroteSAE
from src.sae_train import fit_sae

cache_path = PROJECT_ROOT / "outputs" / "gpt2" / "gpt2_layer8_resid.pt"
max_tokens = 2048


In [ ]:
if not cache_path.exists():
    cache_gpt2_residual_activations(
        ActivationCacheConfig(
            model_name="gpt2",
            layer=8,
            max_tokens=max_tokens,
            batch_size=4,
            sequence_length=64,
            device="cpu",
        ),
        cache_path,
    )
x = load_activation_cache(cache_path)[:max_tokens].to(DEVICE)
print(x.shape, float(x.mean()), float(x.std()))


In [ ]:
expansion_factors = [2, 4]
lambdas = [0.0, 0.5, 1.0, 2.0, 3.0]
train_steps = 300
rows = []
for expansion in expansion_factors:
    width = int(expansion * x.shape[1])
    for lam in lambdas:
        model = VariationalGarroteSAE(
            VGSAEConfig(input_dim=x.shape[1], n_latents=width, lambda_sparsity=lam, beta=1.0)
        )
        fit_sae(model, x, max_steps=train_steps, batch_size=128, lr=1e-3, history_every=100, seed=0)
        obs = vg_sae_observables(model, x)
        rows.append({
            "expansion_factor": expansion,
            "width": width,
            "lambda": lam,
            "mse": obs.mse,
            "rho": obs.rho,
            "entropy": obs.entropy,
            "v_eff": obs.v_eff,
            "dead_fraction": obs.dead_fraction,
            "interference_energy": obs.interference_energy,
            "variance_energy": obs.variance_energy,
        })
df = pd.DataFrame(rows)
for expansion in expansion_factors:
    mask = df["expansion_factor"] == expansion
    df.loc[mask, "susceptibility"] = susceptibility(
        df.loc[mask, "lambda"].to_numpy(dtype=float),
        df.loc[mask, "rho"].to_numpy(dtype=float),
    )
df


In [ ]:
out = OUTPUT_DIR / "exp04_gpt2_layer8"
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "gpt2_layer8_lambda_width_sweep.csv", index=False)

lambda_star = df.loc[df.groupby("width")["v_eff"].idxmax(), ["width", "lambda", "v_eff"]].rename(
    columns={"lambda": "lambda_star"}
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for width, sub in df.groupby("width"):
    sub = sub.sort_values("lambda")
    axes[0].plot(sub["lambda"], sub["v_eff"], marker="o", label=f"N={width}")
    axes[1].plot(sub["lambda"], sub["rho"], marker="o", label=f"N={width}")
axes[0].set_title("Recoverability transition diagnostic")
axes[0].set_xlabel("lambda")
axes[0].set_ylabel("V_eff")
axes[1].set_title("Active density")
axes[1].set_xlabel("lambda")
axes[1].set_ylabel("rho")
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(out / "gpt2_layer8_v_eff_sweep.png", dpi=160)
lambda_star


**Do not oversell this:** if `lambda_star` shifts with width, the result is still informative. It says recoverable feature scale is width-dependent in this setup.
